# 03 Evaluate: วัดผลระดับพิกเซล (เครื่องมือของ Phase 4)

- kernel **`Python (rwd)`**, ต้องมีโมเดลจาก `02_train.ipynb` ก่อน
- ตัววัดหลัก: **precision / recall / IoU ระดับพิกเซล** ของ `weedy_rice` (ดูเหตุผลใน `docs/PLAN.md`) และแสดงค่า instance ของ YOLO คู่กันเสมอ
- ข้อ 2-4 ใช้ **val** เท่านั้น: ใช้เลือกโมเดลและค่า conf **ห้ามนำตัวเลข val ไปเผยแพร่**
- ข้อ 5 คือ **test: ใช้ได้ครั้งเดียว** ต้องได้ OK จากเจ้าของโปรเจกต์ก่อน และสคริปต์จะไม่ยอมรันซ้ำ
- รันซ้ำได้: ถ้ามีผล val อยู่แล้วจะอ่านจากไฟล์ ตั้ง `RERUN = True` เพื่อคำนวณใหม่

## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
PY = sys.executable
if ".venv" not in PY:
    print("WARNING: kernel is not 'Python (rwd)'. Run scripts/setup_env.sh, then Kernel -> Change Kernel.")

from rwd.train import load_train_config
CONFIG = ROOT / "configs/train.yaml"   # train.yaml = โมเดลที่เลือกใช้ | Optional: train_s.yaml, train_s_aug.yaml, train_cofly.yaml
cfg = load_train_config(CONFIG)
RUN = Path(cfg["project"]) / cfg["name"]     # เปลี่ยนเป็นโฟลเดอร์ run อื่นได้ เช่น ROOT / "runs/<name>"
WEIGHTS = RUN / "weights/best.pt"
DATA = Path(cfg["data"])
print("run:", RUN.name, "| weights exist:", WEIGHTS.exists())

## 2. วัดบน val ที่หลายค่า conf

conf ต่ำ = จับวัชพืชได้มากขึ้น (recall สูง) แต่แจ้งผิดมากขึ้น (precision ต่ำ)
ใช้ตารางนี้เลือกค่า conf ที่จะใช้จริง **ก่อน** ไปวัดบน test

In [ ]:
RERUN = False
CONFS = [0.05, 0.1, 0.15, 0.25, 0.4, 0.5]
VAL_JSON = RUN / "eval_val.json"
if RERUN or not VAL_JSON.exists():
    confs = " ".join(str(c) for c in CONFS)
    !"{PY}" -m rwd.evaluate --weights "{WEIGHTS}" --data "{DATA}" --split val --conf {confs}
val = json.loads(VAL_JSON.read_text())
print(f"val: {val['images']} ภาพ  [ใช้เลือกโมเดลเท่านั้น ไม่ใช่ผลที่เผยแพร่ได้]")
print(f"{'conf':>6} {'precision':>10} {'recall':>8} {'IoU':>8} {'IoU/ภาพ (median)':>18}")
for conf, r in val["pixel"].items():
    a = r["all"]
    print(f"{conf:>6} {a['precision']:>10.3f} {a['recall']:>8.3f} {a['iou']:>8.3f} {a['image_iou_median']:>18.3f}")
print("\nค่า instance ของ YOLO (แสดงคู่กันเสมอ):")
for k, v in val["instance_default_conf"].items():
    print(f"  {k:<24} {v}")

In [ ]:
import matplotlib.pyplot as plt

confs = [float(c) for c in val["pixel"]]
fig, ax = plt.subplots(figsize=(7, 4))
for name in ("precision", "recall", "iou"):
    ax.plot(confs, [val["pixel"][c]["all"][name] for c in val["pixel"]], marker="o", label=f"pixel {name}")
ax.set_xlabel("conf"); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend(); ax.set_title("val (model selection only)")
plt.show()

## 3. ผลแยกกลุ่ม (ที่ค่า conf ที่เลือก)

ตั้ง `CONF` เป็นค่าที่จะใช้จริง ต้องเป็นค่าหนึ่งใน `CONFS` ด้านบน
val มีแค่วันเดียวและความสูง 20 ม. ทั้งหมด ตารางนี้จะมีประโยชน์จริงกับ test (มีทั้ง 12 ม. และ 20 ม.)

In [ ]:
CONF = 0.25

def show_groups(report, conf):
    r = report["pixel"][f"{conf:g}"]
    for title, key in (("ความสูงบิน", "by_altitude"), ("วันถ่าย", "by_group")):
        print(title)
        for g, a in r[key].items():
            print(f"  {g:<12} ภาพ={a['images']:>4}  precision={a['precision']}  recall={a['recall']}  IoU={a['iou']}")

show_groups(val, CONF)

## 4. ภาพที่โมเดลพลาดมากที่สุด

**เขียว** = เจอวัชพืชถูกต้อง, **แดง** = วัชพืชที่พลาด (ไม่เจอ), **น้ำเงิน** = แจ้งผิด (ไม่ใช่วัชพืชตาม label)
ดูว่าพลาดแบบไหน: หย่อมเล็ก ขอบหย่อม หรือ label เองที่น่าสงสัย

In [ ]:
import cv2
from ultralytics import YOLO
from rwd.evaluate import error_overlay, label_mask, predict_mask, scores
from rwd.yolo_io import load_data_yaml

def show_worst(report, split, conf, n=6):
    rows = report["per_image"][f"{conf:g}"]
    rows = sorted(rows, key=lambda r: scores(r["tp"], r["fp"], r["fn"])["iou"] or 0)[:n]
    model = YOLO(str(WEIGHTS))
    images_dir = load_data_yaml(DATA).splits[split]
    fig, axes = plt.subplots((n + 2) // 3, 3, figsize=(18, 4.6 * ((n + 2) // 3)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, rows):
        p = images_dir / r["image"]
        pred = predict_mask(model, p, conf, report["imgsz"], cfg.get("device", 0))
        s = scores(r["tp"], r["fp"], r["fn"])
        ax.imshow(error_overlay(cv2.imread(str(p)), pred, label_mask(p, *pred.shape)))
        ax.set_title(f"IoU {s['iou'] or 0:.2f}  recall {s['recall'] or 0:.2f}  {r['image'][13:40]}", fontsize=9)
    plt.tight_layout(); plt.show()

show_worst(val, "val", CONF)

## 5. Phase 4: วัดบน test (ครั้งเดียว)

**อย่ารันจนกว่าจะ:**
1. เลือกโมเดลสุดท้ายและค่า `CONF` จาก val เรียบร้อยแล้ว
2. เจ้าของโปรเจกต์ OK ให้ใช้ test

รันแล้วจะได้ `eval_test.json` ในโฟลเดอร์ run และสคริปต์จะไม่ยอมรันซ้ำกับ run เดิม
ตัวเลขจาก test เท่านั้นที่เผยแพร่ได้ และต้องรายงาน recall คู่กับ precision เสมอ

In [ ]:
OWNER_OK_TO_USE_TEST = False   # เปลี่ยนเป็น True เมื่อพร้อมจริง ๆ เท่านั้น
TEST_JSON = RUN / "eval_test.json"
if TEST_JSON.exists():
    test = json.loads(TEST_JSON.read_text())
    conf = next(iter(test["pixel"]))
    a = test["pixel"][conf]["all"]
    print(f"test ({test['images']} ภาพ) conf {conf}: pixel precision={a['precision']} recall={a['recall']} IoU={a['iou']}")
    print("instance:", test["instance_default_conf"])
    show_groups(test, float(conf))
elif OWNER_OK_TO_USE_TEST:
    !"{PY}" -m rwd.evaluate --weights "{WEIGHTS}" --data "{DATA}" --split test --conf {CONF} --final
    print("เสร็จแล้ว รัน cell นี้อีกครั้งเพื่อดูผลแยกกลุ่ม")
else:
    print("ยังไม่ได้วัดบน test (ตั้งใจ) ตั้ง OWNER_OK_TO_USE_TEST = True เมื่อได้ OK แล้ว")